# Pokrytí mamografickým screeningem: rozšiřující analýzy

Navazuje na revizi `mamograf_analyza.ipynb` a zkoumá otázky, na které tam chyběla data:

| Kap. | Otázka | Nová data |
|---|---|---|
| 1 | Jak velká je skutečná dojížďka a kapacita, když opravíme vykazování řetězců? | NR-04-86 po jednotlivých IČZ |
| 2 | Vysvětluje „gynekologický kanál“ počet gynekologů? | NRPZS (ordinace registrujících gynekologů) |
| 3 | Zvedne otevření pracoviště pokrytí v okrese? | historické seznamy pracovišť mamo.cz (web.archive.org) |
| 4 | Hraje roli složení pojištěnců podle pojišťoven? | (data na úrovni okresů nejsou) |
| 5 | Souvisí pokrytí se stadiem karcinomu při diagnóze? | Národní onkologický registr (NOR) |
| 6 | Jak přesné jsou okresní odhady, hlavně u malých pásem? | small-area model (Fay–Herriot) |
| 7 | Které okresy jsou pod a nad očekáváním? | model M3 z revize |

Všechna data jsou v `data/` (agregáty z velkých zdrojů počítá `utils/priprava_dat.py`, popis v
`DATA.md`). Společné kroky s revizí (standardizace, 2SFCA, Moran, křížová validace) jsou v
`utils/analyza.py`. Analýza je pořád **ekologická**.

In [1]:
import sys
import warnings
from pathlib import Path

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "nacteni.py").exists())
sys.path.insert(0, str(BASE))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

from utils import analyza as an
from utils import nacteni as nac
from utils.vzhled import AXIS, BLUES, DIV, GRID, INK2, MUTED, S1, S2, SURFACE, cz, layout, mapa_geo, skala

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", module="statsmodels")
pd.set_option("display.max_columns", 40)
pio.renderers.default = "plotly_mimetype+notebook_connected"

OUT = BASE / "outputs" / "rozsireni"
OUT.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(2026)

V = nac.vse()
D, X, GEO, CENTROID, centra, OKRES = (V[k] for k in ("D", "X", "GEO", "CENTROID", "centra", "OKRES"))
LAUS = sorted(CENTROID)
X = X.set_index("lau").reindex(LAUS)
S, P, STANDARD = an.pokryti(D, LAUS)
X["prev_index"] = an.prev_index(X)
icz, gyn_icz = nac.nrhzs_icz()
print(f"{len(LAUS)} okresů · {icz.ICZ.nunique()} IČZ se screeningovou mamografií · "
      f"{gyn_icz.ICZ.nunique()} IČZ s preventivní gynekologií")

77 okresů · 126 IČZ se screeningovou mamografií · 1610 IČZ s preventivní gynekologií


## 1 · Toky po IČZ: skutečná dojížďka a kapacita

Revize (kap. 3.1) našla 7 okresů, kde mamo.cz uvádí pracoviště, ale NRHZS v nich nevykazuje
žádnou mamografii. Data po jednotlivých IČZ ukazují proč: jeden kód IČZ vykazuje za **více
fyzických pracovišť** téhož poskytovatele (např. EUC vykazuje Bílinu i Českou Lípu pod IČZ v Ústí
nad Labem). Kód tedy nejde jednoduše přesunout, je potřeba ho rozdělit.

**Pravidlo:** v těchto 7 okresech počítáme vyšetření u IČZ, kam z okresu míří nejvíc žen, jako
vyšetření v místě, a stejná vyšetření přičítáme k výkonu tamního pracoviště. Je to heuristika:
předpokládá, že ženy z okresu s vlastním pracovištěm řetězce chodí hlavně na něj.

Kontrola: toky po IČZ sečtené po okresech se přesně shodují s okresními toky, se kterými
pracovala revize.

In [2]:
R = icz[icz.rok.isin(nac.ROKY_V)]
toky = nac.csv("nrhzs_mamografie_toky.csv")
a = R.groupby(["rok", "okres_pacient", "okres_ICZ"]).mnozstvi.sum()
b = toky[toky.rok.isin(nac.ROKY_V)].set_index(["rok", "z_lau", "cil_lau"]).mnozstvi
b.index.names = a.index.names
assert np.allclose(pd.concat([a, b], axis=1).fillna(0).diff(axis=1).iloc[:, 1], 0), "IČZ ≠ okresní toky"

celkem = R.groupby("okres_pacient").mnozstvi.sum().reindex(LAUS)
doma = R[R.okres_ICZ == R.okres_pacient].groupby("okres_pacient").mnozstvi.sum().reindex(LAUS).fillna(0)
FANTOM = sorted(set(centra.obec_lau) - set(R.okres_ICZ))
HLAVNI_ICZ = {}
for l in FANTOM:
    s = R[R.okres_pacient == l].groupby("ICZ").mnozstvi.sum()
    HLAVNI_ICZ[l] = s.idxmax()
    doma[l] += s.max()
X["dojizdka_kor"] = 100 * (1 - doma / celkem)

R2 = R.assign(misto=R.okres_ICZ)
for l, i in HLAVNI_ICZ.items():
    R2.loc[(R2.ICZ == i) & (R2.okres_pacient == l), "misto"] = l
VYKON = R2.groupby("misto").mnozstvi.sum() / len(nac.ROKY_V)          # vyšetření za rok v okrese
assert set(VYKON.index) <= set(centra.obec_lau), "výkon v okrese bez pracoviště"

oprava = pd.DataFrame({"okres": X.okres[FANTOM], "IČZ s nejvíc ženami": pd.Series(HLAVNI_ICZ),
                       "vykazuje v": [OKRES[R[R.ICZ == HLAVNI_ICZ[l]].okres_ICZ.iloc[0]] for l in FANTOM],
                       "dojížďka původně %": X.dojizdka_mamo_pct[FANTOM],
                       "dojížďka opravená %": X.dojizdka_kor[FANTOM]})
oprava.round(1)

,okres,IČZ s nejvíc ženami,vykazuje v,dojížďka původně %,dojížďka opravená %
CZ0201,Benešov,14214000,Praha,100.0,13.0
CZ0327,Tachov,44101000,Plzeň-město,100.0,19.2
CZ0426,Teplice,59370000,Ústí nad Labem,100.0,7.5
CZ0511,Česká Lípa,59370000,Ústí nad Labem,100.0,18.7
CZ0711,Jeseník,93201000,Šumperk,100.0,51.4
CZ0723,Vsetín,88631000,Nový Jičín,100.0,5.2
CZ0801,Bruntál,90001000,Opava,100.0,44.0


Pět ze sedmi okresů má po opravě dojížďku 5–19 %, jako běžný okres s vlastním pracovištěm.
**Jeseník a Bruntál** ale i po opravě posílají mimo okres skoro polovinu žen. Tamní pracoviště
(Jeseník, Krnov) jsou malá a ženy jezdí do Šumperka, Opavy a Olomouce.

### 1.1 Kapacita: pracoviště v dosahu vs. skutečně provedená vyšetření

Revize měřila nabídku počtem akreditovaných pracovišť v dosahu (2SFCA) a nenašla žádný vztah.
Teď můžeme nabídku vážit **skutečným výkonem** pracovišť (vyšetření za rok, rozdělená mezi
pracoviště okresu). Výsledek je počet vyšetření v dosahu na 100 žen 45–74 let.

**Pozor na kruhový argument:** provedená vyšetření jsou z velké části právě to pokrytí, které
vysvětlujeme. Kde ženy chodí víc, tam pracoviště víc vyšetří. Výkonová míra proto ukazuje spíš
realizovanou poptávku než nabídku a kauzálně ji číst nelze.

In [3]:
popt = D[D.band.astype(str).isin(nac.BANDS[:6]) & D.rok.isin(nac.ROKY_V)].groupby("lau").zeny.mean().reindex(LAUS)
clon = np.array([CENTROID[l][0] for l in LAUS]); clat = np.array([CENTROID[l][1] for l in LAUS])
DIST = nac.haversine(clon[:, None], clat[:, None], centra.lon.values[None, :], centra.lat.values[None, :])
kapacita = centra.obec_lau.map(VYKON / centra.obec_lau.value_counts()).fillna(0).values
rows = []
for d0 in (20, 30, 45):
    X[f"sfca_{d0}"] = an.sfca(DIST, popt, d0)
    X[f"sfca_vykon_{d0}"] = an.sfca(DIST, popt, d0, nabidka=kapacita, nasobek=100)
    for c, jm in [(f"sfca_{d0}", "pracoviště v dosahu"), (f"sfca_vykon_{d0}", "provedená vyšetření v dosahu")]:
        for cil, cjm in [("m_std", "standardizované 45+"), ("m_5069", "50–69 let")]:
            r, p = stats.pearsonr(X[c], S[cil])
            rows.append({"dosah km": d0, "míra": jm, "cíl": cjm, "r": r, "p": p})
for cil, cjm in [("m_std", "standardizované 45+"), ("m_5069", "50–69 let")]:
    for c, jm in [("dojizdka_mamo_pct", "dojížďka (původní)"), ("dojizdka_kor", "dojížďka (opravená)")]:
        r, p = stats.pearsonr(X[c], S[cil])
        rows.append({"dosah km": None, "míra": jm, "cíl": cjm, "r": r, "p": p})
KAP = pd.DataFrame(rows)
KAP.to_csv(OUT / "kapacita_dojizdka.csv", index=False)
KAP.pivot_table(index=["míra", "dosah km"], columns="cíl", values="r", dropna=False).dropna(how="all").round(3)

cíl                                    50–69 let  standardizované 45+
míra                         dosah km                                
dojížďka (opravená)          NaN          -0.209               -0.302
dojížďka (původní)           NaN          -0.252               -0.340
pracoviště v dosahu          20.0         -0.004                0.055
                             30.0         -0.032                0.025
                             45.0         -0.006                0.041
provedená vyšetření v dosahu 20.0          0.269                0.380
                             30.0          0.215                0.317
                             45.0          0.192                0.269

**Jak to číst.**

* **Měřená dojížďka po opravě drží:** okresy, odkud ženy víc jezdí jinam, mají nižší pokrytí
  (r ≈ −0,30 u standardizovaného 45+, u 50–69 slaběji). Revize to ukázala vyřazením sporných
  okresů, teď to platí i s jejich opravenými hodnotami.
* **Počet pracovišť v dosahu s pokrytím nesouvisí**, výkon pracovišť v dosahu ano (r ≈ 0,3–0,4).
  Rozdíl je čekaný: výkon je z velké části samo pokrytí. Data tak neumí oddělit, jestli chybí
  kapacita, nebo zájem. K tomu by byly potřeba objednací lhůty (mamo.cz je publikovalo do roku
  2019, viz kap. 3).

### 1.2 Objednací lhůty

Výkon pracovišť od zájmu neoddělíme, objednací lhůty ale ano: měří, jak těžké je se k vyšetření
dostat. Archivované seznamy pracovišť na mamo.cz (web.archive.org) uváděly u každého pracoviště
„objednací lhůtu“ s datem údaje, dohromady 419 údajů z let 2013–2019 (6 snímků, 67–73 pracovišť).

**Lhůta, kterou ženy z okresu zažívají**, je průměr lhůt pracovišť vážený tím, kam ženy z okresu
na mamograf skutečně jezdí (toky NRHZS 2020–2024 po opravě z kap. 1.0). Zachytí tak i okresy bez
vlastního pracoviště. Dva testy:

* **mezi okresy:** souvisí dlouhodobě delší lhůty s nižším pokrytím, i po zohlednění modelu M3?
* **uvnitř okresů v čase:** sníží prodloužení lhůty v daném roce pokrytí (pevné efekty okresu a roku)?

Pozor na obrácený vztah: dlouhá lhůta může znamenat i velký zájem. Pak by vycházela s pokrytím kladně.

In [4]:
cek = nac.cekaci_doby()
TOKY_MISTO = R2.groupby(["okres_pacient", "misto"]).mnozstvi.sum().rename("n").reset_index()

def zazita_lhuta(lhuta_okresu):
    '''Průměr lhůt pracovišť vážený toky žen z okresu (kam skutečně jezdí).'''
    t = TOKY_MISTO.assign(w=TOKY_MISTO.misto.map(lhuta_okresu)).dropna(subset=["w"])
    return t.groupby("okres_pacient").apply(lambda g: np.average(g.w, weights=g.n), include_groups=False).reindex(LAUS)

X["cekani_dni"] = zazita_lhuta(cek.groupby("lau").cekani_dni.mean())
ROCNI_ = an.rocni_std(D, STANDARD)
pokr_1419 = 2 * ROCNI_[ROCNI_.rok.between(2014, 2019)].groupby("lau").m1.mean().reindex(LAUS)

rows = []
for jm, y in [("pokrytí 2014–2019 (stejné období jako lhůty)", pokr_1419),
              ("standardizované pokrytí 45+, 2020–2024", S.m_std), ("pokrytí 50–69 let, 2020–2024", S.m_5069)]:
    r, p = stats.pearsonr(X.cekani_dni, y)
    rows.append({"pokrytí": jm, "r": r, "p": p, "Spearman ρ": stats.spearmanr(X.cekani_dni, y)[0]})
CEK = pd.DataFrame(rows).set_index("pokrytí")
souvis = {jm: X.cekani_dni.corr(X[c]) for c, jm in [("prev_index", "index prevence"), ("sfca_30", "pracoviště v dosahu"),
          ("ma_mamocentrum", "vlastní pracoviště"), ("dojizdka_kor", "dojížďka"), ("nezam_zeny_pct", "nezaměstnanost žen")]}

M3 = an.M1 + ["sfca_30", "ma_mamocentrum", "prev_index"]
o_cek, _ = an.ols_std(S.m_std, X, M3 + ["cekani_dni"])
cv_m3 = an.cv_r2(X[M3].values, S.m_std.values)[0]
cv_m3c = an.cv_r2(X[M3 + ["cekani_dni"]].values, S.m_std.values)[0]

pan = []
for rok, g in cek.groupby("rok"):
    pan.append(pd.DataFrame({"lau": LAUS, "rok": rok, "cekani": zazita_lhuta(g.groupby("lau").cekani_dni.mean()).values}))
pan = pd.concat(pan).groupby(["lau", "rok"]).cekani.mean().reset_index()
FE_CEK = {}
for jm, posun in [("stejný rok", 0), ("následující rok", 1)]:
    q = pan.assign(rok=pan.rok + posun).merge(ROCNI_[["lau", "rok", "m1"]], on=["lau", "rok"]).dropna()
    m = smf.ols("m1 ~ cekani + C(lau) + C(rok)", q).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(q.lau)[0]})
    FE_CEK[jm] = (10 * m.params["cekani"], 10 * m.bse["cekani"], m.pvalues["cekani"], int(m.nobs))

print(f"objednací lhůta: {len(cek)} údajů z {cek.snimek.nunique()} snímků; medián podle roku údaje "
      + ", ".join(f"{k} {v:.0f} dní" for k, v in cek.groupby('rok').cekani_dni.median().items()))
print(f"zažitá lhůta po okresech: medián {X.cekani_dni.median():.1f} dne, rozpětí {X.cekani_dni.min():.1f}–{X.cekani_dni.max():.1f}")
print("souvislost lhůty s ostatními ukazateli: " + ", ".join(f"{k} {v:+.2f}" for k, v in souvis.items()))
print(f"M3 + lhůta: β = {o_cek.params[-1]:+.2f} p. b. na SD (p = {o_cek.pvalues[-1]:.3f}); "
      f"R² mimo vzorek {cv_m3:.3f} → {cv_m3c:.3f}")
for jm, (b, se, p, n) in FE_CEK.items():
    print(f"pevné efekty, {jm}: +10 dní → {b:+.2f} p. b. (SE {se:.2f}, p = {p:.2f}, n = {n})")
CEK.round(3)

objednací lhůta: 419 údajů z 6 snímků; medián podle roku údaje 2013 16 dní, 2015 8 dní, 2016 8 dní, 2017 10 dní, 2018 10 dní
zažitá lhůta po okresech: medián 12.3 dne, rozpětí 4.5–39.6
souvislost lhůty s ostatními ukazateli: index prevence +0.02, pracoviště v dosahu -0.11, vlastní pracoviště +0.02, dojížďka +0.02, nezaměstnanost žen +0.38
M3 + lhůta: β = -1.13 p. b. na SD (p = 0.013); R² mimo vzorek 0.288 → 0.335
pevné efekty, stejný rok: +10 dní → -0.02 p. b. (SE 0.12, p = 0.90, n = 385)
pevné efekty, následující rok: +10 dní → +0.11 p. b. (SE 0.11, p = 0.31, n = 385)


,r,p,Spearman ρ
pokrytí,,,
pokrytí 2014–2019 (stejné období jako lhůty),-0.321,0.004,-0.289
"standardizované pokrytí 45+, 2020–2024",-0.369,0.001,-0.324
"pokrytí 50–69 let, 2020–2024",-0.366,0.001,-0.340


In [5]:
cf = go.Figure(go.Scatter(
    x=X.cekani_dni, y=S.m_std, mode="markers",
    marker=dict(size=9, color=S1, line=dict(color=SURFACE, width=1.5)),
    customdata=np.c_[X.okres, X.kraj],
    hovertemplate="<b>%{customdata[0]}</b> · %{customdata[1]}<br>lhůta %{x:.1f} dne<br>pokrytí %{y:.1f} %<extra></extra>"))
k_ = np.polyfit(X.cekani_dni, S.m_std, 1); xs = np.array([X.cekani_dni.min(), X.cekani_dni.max()])
cf.add_trace(go.Scatter(x=xs, y=np.polyval(k_, xs), mode="lines", line=dict(color=MUTED, width=2), hoverinfo="skip"))
r0 = CEK.iloc[1]
cf.update_layout(**layout("Kde se na mamograf dlouho čeká, chodí méně žen",
                          f"objednací lhůta 2013–2019 vážená tím, kam ženy z okresu jezdí · "
                          f"standardizované pokrytí 2020–2024 · r = {cz(r0.r, 2)}"))
cf.update_layout(height=480, showlegend=False, xaxis_title="zažitá objednací lhůta [dny]",
                 yaxis_title="standardizované pokrytí 45+ [%]", margin=dict(l=60, r=30, t=90, b=60))
cf.write_html(OUT / "objednaci_lhuty.html", include_plotlyjs="cdn")
cf

**Jak to číst.**

* **Delší lhůty jdou s nižším pokrytím** (r ≈ −0,3 až −0,37, p < 0,01), a to i u pokrytí ze
  stejného období jako lhůty. Kladný vztah, který by čekal výklad „dlouho se čeká, protože je
  zájem“, data nepodporují.
* **Lhůta je nezávislý ukazatel.** S indexem prevence, kapacitou v dosahu, vlastním pracovištěm ani
  dojížďkou skoro nesouvisí (|r| ≤ 0,1); souvisí s nezaměstnaností. Vztah drží i po zohlednění
  modelu M3 (asi −1,1 p. b. na směrodatnou odchylku lhůty, tj. na každých +8 dní) a zlepší
  předpověď mimo vzorek.
* **Uvnitř okresů v čase vztah nevidíme**: když se lhůta v okrese prodlouží, pokrytí ten ani
  další rok neklesne (+10 dní → −0,02 p. b., SE 0,12). Typický meziroční výkyv lhůty v okrese je
  asi 6 dní a lhůta je jeden údaj za rok, takže test malý účinek přehlédnout může.
* **Výklad:** okresy s chronicky dlouhými lhůtami mají nižší pokrytí, ale delší čekání samo pokrytí
  nesnižuje. Souvislost mezi okresy tedy spíš odráží trvalé vlastnosti okresů (chronický nedostatek
  kapacity, organizace péče v kraji, chudoba; lhůta souvisí s nezaměstnaností žen, r = 0,38) než přímý
  účinek čekání. Příčinu tato data neurčí.
* Pro zadání: objednací lhůta patří mezi **hlídané ukazatele** experimentu a kampaní, které zvyšují
  poptávku (pevný termín, připomínky), ne mezi doložené páky. Lhůty po roce 2019 ve veřejných datech
  nejsou; VZP by je mohla sbírat přes propojení rezervačních systémů s Moje VZP.

## 2 · Nabídka gynekologů

Nejsilnějším korelátem pokrytí je preventivní gynekologická prohlídka (revize, kap. 4 a 6).
Pokud jde o **dostupnost gynekologů**, mělo by pokrytí růst s jejich hustotou. Měříme ji
dvakrát:

* **ordinace registrujících gynekologů** na 10 000 žen (NRPZS, místa poskytování péče, stav 10/2026),
* **IČZ vykazující preventivní prohlídku** (výkon 63050, aspoň 50 pacientek za rok) na 10 000
  žen, průměr 2020–2024 (NR-04-86).

Pak se ptáme, jestli nabídka souvisí s gynekologickou prevencí, s pokrytím, a jestli pokrytí
ovlivňuje *přes* preventivní chování (ekologická mediace: celkový efekt, efekt nabídky na index
prevence a přímý efekt po jeho zahrnutí, vše s kontrolou socioekonomiky M1).

In [6]:
gyn = nac.gyn_okresy().set_index("lau").reindex(LAUS)
X["gyn_ord_10k"] = 1e4 * gyn.gyn_ordinace / X.zeny_celkem
gi = (gyn_icz[gyn_icz.rok.isin(nac.ROKY_V) & (gyn_icz.pacienti >= 50)]
      .groupby(["rok", "okres_ICZ"]).ICZ.nunique().groupby("okres_ICZ").mean().reindex(LAUS))
X["gyn_icz_10k"] = 1e4 * gi / X.zeny_celkem
POPIS = {**nac.POPIS, "prev_index": ("index preventivního chování", "z-skóre"),
         "gyn_ord_10k": ("ordinace gynekologů na 10 000 žen", ""),
         "gyn_icz_10k": ("IČZ s gyn. prevencí na 10 000 žen", ""),
         "dojizdka_kor": ("dojížďka na mamograf, opravená", "%"),
         "sfca_30": ("pracoviště v dosahu 30 km", ""), "sfca_vykon_30": ("vyšetření v dosahu 30 km", "")}

def std_ols(y, cols):
    o, _ = an.ols_std(y, X, cols)
    return o

rows = []
for c in ["gyn_ord_10k", "gyn_icz_10k"]:
    for cil, jm in [(X.gyn_prevence_pct, "gyn. prevence"), (X.prev_index, "index prevence"),
                    (S.m_std, "pokrytí 45+ (std.)"), (S.m_5069, "pokrytí 50–69")]:
        r, p = stats.pearsonr(X[c], cil)
        rows.append({"nabídka": POPIS[c][0], "s čím": jm, "r": r, "p": p})
GYN = pd.DataFrame(rows)

med = []
for c in ["gyn_ord_10k", "gyn_icz_10k"]:
    tot = std_ols(S.m_std, [c] + an.M1); a_ = std_ols(X.prev_index, [c] + an.M1)
    pri = std_ols(S.m_std, [c, "prev_index"] + an.M1)
    med.append({"nabídka": POPIS[c][0], "celkový efekt β": tot.params[1], "p": tot.pvalues[1],
                "nabídka → index prevence β": a_.params[1], "p ": a_.pvalues[1],
                "přímý efekt β": pri.params[1], "p  ": pri.pvalues[1]})
MED = pd.DataFrame(med)
GYN.to_csv(OUT / "gynekologove.csv", index=False)
print(f"ordinace gynekologů: {int(gyn.gyn_ordinace.sum())} v ČR, medián {X.gyn_ord_10k.median():.1f} na 10 000 žen "
      f"(rozpětí {X.gyn_ord_10k.min():.1f}–{X.gyn_ord_10k.max():.1f}); "
      f"korelace obou měr r = {X.gyn_ord_10k.corr(X.gyn_icz_10k):.2f}")
display(GYN.pivot(index="s čím", columns="nabídka", values="r").round(3))
MED.round(3)

ordinace gynekologů: 1629 v ČR, medián 2.9 na 10 000 žen (rozpětí 1.5–5.7); korelace obou měr r = 0.53


nabídka,IČZ s gyn. prevencí na 10 000 žen,ordinace gynekologů na 10 000 žen
s čím,,
gyn. prevence,0.087,0.138
index prevence,0.175,0.219
pokrytí 45+ (std.),0.129,0.054
pokrytí 50–69,0.098,0.015


,nabídka,celkový efekt β,p,nabídka → index prevence β,p,přímý efekt β,p
0,ordinace gynekologů na 10 000 žen,0.289,0.612,0.125,0.191,-0.113,0.823
1,IČZ s gyn. prevencí na 10 000 žen,0.661,0.322,0.084,0.391,0.399,0.493


**Jak to číst.** Hustota gynekologů s pokrytím mamografií **nesouvisí** (|r| ≤ 0,13) a s
gynekologickou prevencí jen slabě (index prevence r ≈ 0,2, na hraně průkaznosti). Mediace
nic neukazuje. „Gynekologický kanál“ z revize tedy není otázka toho, **kolik** je gynekologů,
ale toho, **co dělají**: jestli při prohlídce posílají ženy na mamograf a jestli ženy na
prohlídky chodí. To je dobrá zpráva pro intervence: doporučení od gynekologa se dá ovlivnit
bez nových ordinací (srov. `[GP_ENDORSE2011]` v `PAPERS.md`).

## 3 · Otevření pracoviště: případová studie

Seznamy akreditovaných pracovišť z let 2008–2019 (snímky mamo.cz ve web.archive.org) a dnešní
seznam dávají počet pracovišť v každém okrese a roce. Síť je **velmi stabilní**: 66 pracovišť
v roce 2008, 73 v roce 2019, 75 dnes. Vlastní první pracoviště za tu dobu získaly jen tři okresy:
Teplice (2016, EUC Bílina), Jeseník a Bruntál (mezi 2019 a 2026; podle toků po IČZ ale
tamní pracoviště vykazovala už v roce 2019, takže jde spíš o pozdější akreditaci). Vsetín
v seznamu chyběl v letech 2010–2011.

Formální event study s jedním použitelným případem nedává smysl. Teplice proto porovnáváme se
zbytkem Ústeckého kraje (rozdíl v rozdílech) a **placebo testem**: stejný odhad pro každý jiný
okres, jako by pracoviště dostal v roce 2016. Teplice je průkazné, jen pokud vyjde výjimečně
proti rozložení placebo efektů.

In [7]:
hist = nac.centra_historie().pivot(index="lau", columns="rok", values="pracovist").reindex(LAUS).fillna(0)
ROCNI = an.rocni_std(D, STANDARD)
wide = ROCNI.pivot(index="lau", columns="rok", values="m1")

def did(l, rok0=2016, okno=(2010, 2024)):
    '''Rozdíl v rozdílech: okres l proti zbytku svého kraje, po (od rok0) minus před.'''
    ctrl = [x for x in LAUS if x[:5] == l[:5] and x != l]
    gap = wide.loc[l] - wide.loc[ctrl].mean()
    gap = gap[(gap.index >= okno[0]) & (gap.index <= okno[1])]
    return gap[gap.index >= rok0].mean() - gap[gap.index < rok0].mean(), gap

efekt_tep, gap_tep = did("CZ0426")
placebo = pd.Series({l: did(l)[0] for l in LAUS if l != "CZ0426" and l != "CZ0100"})
p_placebo = (1 + (placebo >= efekt_tep).sum()) / (len(placebo) + 1)
print("pracoviště v okresech podle roku:", hist.sum().astype(int).to_dict())
print(f"Teplice: rozdíl proti zbytku Ústeckého kraje před 2016 {gap_tep[gap_tep.index < 2016].mean():+.1f} p. b., "
      f"od 2016 {gap_tep[gap_tep.index >= 2016].mean():+.1f} p. b. → efekt {efekt_tep:+.1f} p. b.")
print(f"placebo (76 okresů): efekt ≥ Teplice u {(placebo >= efekt_tep).sum()} okresů, p = {p_placebo:.2f}")

pracoviště v okresech podle roku: {2008: 66, 2009: 66, 2010: 66, 2011: 67, 2013: 68, 2015: 68, 2016: 69, 2017: 69, 2018: 73, 2019: 73, 2026: 75}
Teplice: rozdíl proti zbytku Ústeckého kraje před 2016 -2.5 p. b., od 2016 -0.3 p. b. → efekt +2.2 p. b.
placebo (76 okresů): efekt ≥ Teplice u 6 okresů, p = 0.09


In [8]:
tf = go.Figure()
tf.add_trace(go.Scatter(x=gap_tep.index, y=gap_tep.values, mode="lines+markers", name="Teplice − zbytek kraje",
                        line=dict(color=S1, width=3)))
_, gap_vs = did("CZ0723", rok0=2010)
tf.add_trace(go.Scatter(x=gap_vs.index, y=gap_vs.values, mode="lines+markers", name="Vsetín − zbytek kraje",
                        line=dict(color=S2, width=2, dash="dot")))
tf.add_vline(x=2015.5, line=dict(color=AXIS, dash="dash"), annotation_text="Teplice: první pracoviště",
             annotation_position="top left")
tf.add_vrect(x0=2009.5, x1=2011.5, fillcolor=GRID, opacity=0.5, line_width=0,
             annotation_text="Vsetín bez pracoviště", annotation_position="bottom left")
tf.add_hline(y=0, line=dict(color=AXIS, width=1))
tf.update_layout(**layout("Teplice po otevření pracoviště dohnaly zbytek kraje, ale pomalu",
                          f"roční standardizované pokrytí 45+, rozdíl proti průměru ostatních okresů kraje · "
                          f"efekt {cz(efekt_tep)} p. b., placebo p = {cz(p_placebo, 2)}"))
tf.update_layout(height=460, yaxis_title="rozdíl [p. b.]", legend=dict(orientation="h", y=-0.15, x=0),
                 margin=dict(l=60, r=30, t=90, b=80))
tf.write_html(OUT / "teplice.html", include_plotlyjs="cdn")
tf

**Jak to číst.** Teplice zaostávaly za zbytkem kraje o 2–3 p. b. a od otevření pracoviště v Bílině
se rozdíl postupně zavřel, plně až kolem roku 2020. V placebo testu má stejně velkou nebo větší
změnu jen 6 ze 76 okresů (p ≈ 0,09). S **jedním případem** je to ale doklad slabý: ve stejné
době mohlo v Teplicích působit cokoli jiného. U Vsetína dvouletá absence v seznamu na pokrytí stopu nezanechala
(pracoviště nejspíš fungovalo dál, jen bez akreditace). Silnější test by potřeboval víc otevření,
tedy jemnější data (data pracovišť po IČZ jsou až od 2019).

## 4 · Pojišťovny: datová mezera

Zvaní na screening posílají zdravotní pojišťovny a každá to dělá trochu jinak. Krajský vzorec
u žen 50–69 (revize, kap. 2) by proto mohl odrážet **složení pojištěnců**: VZP má v různých
krajích různý podíl, regionální pojišťovny (RBP na severní Moravě, ZP MV, ČPZP, OZP) jsou
soustředěné.

**Otevřená data to neumožňují.** Katalog data.gov.cz (10/2026) má výkony podle pojišťoven jen
za celou ČR (ÚZIS NR-04-23, NR-04-98) a finanční údaje pojišťoven (ČSÚ FIN02Q). Počty pojištěnců
podle pojišťovny a okresu ani kraje volně dostupné nejsou. K testu by bylo potřeba:

* podíl pojištěnek 45+ podle pojišťovny a okresu (mají ho pojišťovny a ÚZIS, NRHZS),
* ideálně počty odeslaných pozvánek a účast po pozvánce podle pojišťovny a okresu.

To je konkrétní žádost na VZP pro hackathon.

## 5 · Stadium karcinomu prsu při diagnóze

Pokud screening funguje, má s vyšším pokrytím klesat podíl karcinomů zachycených pozdě
(stadium III–IV). Národní onkologický registr (NOR) dává případy C50 u žen podle roku diagnózy,
**kraje**, věku a stadia (okresy v otevřených datech nejsou). Bereme ženy 45–74 let a podíl
stadia III–IV mezi případy se známým stadiem (I–IV).

Tři pohledy: celostátní vývoj, panel 14 krajů × 2012–2024 s pevnými efekty kraje a roku
(pokrytí za dva předchozí roky) a průřez kraji.

In [9]:
nor = nac.nor_c50()
inc = nor[(nor.ukazatel == "incidence") & nor.vek_od.between(45, 74)].copy()
inc["pozde"] = inc.stadium.isin(["3", "4"]) * inc.n
inc["znamo"] = inc.stadium.isin(["1", "2", "3", "4"]) * inc.n
narod = inc.groupby("rok")[["pozde", "znamo", "n"]].sum()
narod["pozde_pct"] = 100 * narod.pozde / narod.znamo
narod["neznamo_pct"] = 100 * (1 - narod.znamo / narod.n)

KR = inc.groupby(["kraj", "rok"])[["pozde", "znamo"]].sum().reset_index()
KR["pozde_pct"] = 100 * KR.pozde / KR.znamo
zeny_r = D[D.band != nac.TOTAL].groupby(["lau", "rok"]).zeny.sum().rename("zeny").reset_index()
RK = ROCNI.merge(zeny_r, on=["lau", "rok"]).assign(kraj=lambda t: t.lau.str[:5])
RK = (RK.groupby(["kraj", "rok"]).apply(lambda g: np.average(g.m1, weights=g.zeny), include_groups=False)
        .rename("m1").reset_index().sort_values(["kraj", "rok"]))
RK["m2_pred"] = RK.groupby("kraj").m1.transform(lambda s: s.shift(1) + s.shift(2))
Q = KR.merge(RK, on=["kraj", "rok"]).dropna()
Q = Q[Q.rok.between(2012, 2024)]
fe = smf.wls("pozde_pct ~ m2_pred + C(kraj) + C(rok)", Q, weights=Q.znamo).fit(
    cov_type="cluster", cov_kwds={"groups": pd.factorize(Q.kraj)[0]})
pr = Q.groupby("kraj").apply(lambda g: pd.Series({
    "pozde_pct": 100 * g[g.rok >= 2017].pozde.sum() / g[g.rok >= 2017].znamo.sum(),
    "pokryti_2y": 2 * g[g.rok.between(2015, 2022)].m1.mean()}), include_groups=False)
r_pr = stats.pearsonr(pr.pokryti_2y, pr.pozde_pct)
print(f"celostátně: stadium III–IV {narod.pozde_pct.loc[1995]:.1f} % (1995) → {narod.pozde_pct.loc[2002]:.1f} % (2002, "
      f"start programu) → {narod.pozde_pct.loc[2024]:.1f} % (2024); neznámé stadium {narod.neznamo_pct.loc[2017:].mean():.1f} % (2017–24)")
print(f"panel krajů: +1 p. b. pokrytí za 2 roky → {fe.params['m2_pred']:+.2f} p. b. stadia III–IV "
      f"(SE {fe.bse['m2_pred']:.2f}, p = {fe.pvalues['m2_pred']:.2f}, n = {int(fe.nobs)})")
print(f"průřez 14 krajů: r = {r_pr[0]:.2f} (p = {r_pr[1]:.2f})")

celostátně: stadium III–IV 28.8 % (1995) → 19.8 % (2002, start programu) → 13.3 % (2024); neznámé stadium 3.5 % (2017–24)
panel krajů: +1 p. b. pokrytí za 2 roky → +0.13 p. b. stadia III–IV (SE 0.17, p = 0.45, n = 182)
průřez 14 krajů: r = 0.07 (p = 0.82)


In [10]:
nf = go.Figure()
rr = narod.loc[1990:]
nf.add_trace(go.Scatter(x=rr.index, y=rr.pozde_pct, mode="lines+markers", name="stadium III–IV mezi známými [%]",
                        line=dict(color=S2, width=3)))
cr = ROCNI.merge(zeny_r, on=["lau", "rok"]).groupby("rok").apply(
    lambda g: 2 * np.average(g.m1, weights=g.zeny), include_groups=False)
nf.add_trace(go.Scatter(x=cr.index, y=cr.values, mode="lines+markers", yaxis="y2",
                        name="pokrytí mamografií 45+ za 2 roky [%]", line=dict(color=S1, width=2)))
nf.add_vline(x=2002, line=dict(color=AXIS, dash="dash"), annotation_text="start programu",
             annotation_position="top right")
nf.update_layout(**layout("Podíl pozdě zachycených karcinomů prsu se od startu screeningu snížil",
                          "ženy 45–74 let, karcinom prsu (C50), NOR · pokrytí na pravé ose (data od 2010)"))
nf.update_layout(height=470, yaxis=dict(title="stadium III–IV [%]", range=[0, 32], gridcolor=GRID),
                 yaxis2=dict(overlaying="y", side="right", title="pokrytí [%]", range=[0, 64], showgrid=False),
                 legend=dict(orientation="h", y=-0.15, x=0), margin=dict(l=60, r=60, t=90, b=80))
nf.write_html(OUT / "stadium.html", include_plotlyjs="cdn")
nf

**Jak to číst.**

* **Celostátně** klesl podíl karcinomů ve stadiu III–IV z téměř 30 % v polovině 90. let na
  13 % v roce 2024. Velká část poklesu ale přišla **před startem programu** (1995 → 2002 z 29 %
  na 20 %, oportunní mamografie a lepší diagnostika), po něm pokles pokračoval pomaleji (na 13 %).
  Je to v souladu s přínosem screeningu, ale ne důkaz.
* **Mezi kraji vztah nevidíme**: ani v panelu s pevnými efekty, ani v průřezu. Kraje se v
  pokrytí liší jen o ~12 p. b. (48–60 %) a 14 jednotek s ~350 případy ročně na kraj na takový
  rozdíl nestačí. Navíc jde o ekologické srovnání: pozdní stadia mohou mít hlavně ženy, které na
  screening nechodí, a to agregát kraje neukáže.
* Pro slibovaný „posun stadia“ v ekonomickém modelu to znamená:
  parametry z literatury (`[EUROSCREEN2012]`) zůstávají, krajská data je nepotvrdí ani nevyvrátí.

## 6 · Small-area model: jak přesné jsou okresní odhady

Okresní pokrytí je podíl z konečného počtu žen. V malém okrese a úzkém věkovém pásmu (85+ v
Jeseníku jsou stovky žen) může být odhad zašuměný a mapy pak ukazují šum. Standardní řešení je
**Fay–Herriotův model**: přímý odhad každého okresu se „půjčí sílu“ od sousedů a kraje podle
toho, jak je přesný.

$$y_i = \beta_0 + s_i + k_{c(i)} + e_i,\qquad e_i \sim N(0, v_i)$$

* $s$ je prostorový efekt (Leroux CAR: směs sousedské struktury a nezávislého šumu, váha λ),
* $k$ je efekt kraje, $v_i$ výběrový rozptyl přímého odhadu (binomický, z počtu žen v pásmu),
* parametry odhadujeme REML (`utils/analyza.fay_herriot`), pro každé pásmo zvlášť.

Kontrola: při $v_i \to 0$ model vrací přímé odhady beze změny.

In [11]:
W = an.sousedstvi(GEO, LAUS)
P["neff"] = P.zeny / len(nac.ROKY_V)                  # průměrný roční počet žen v pásmu
rows, FH = [], {}
for b in nac.BANDS + ["50–69"]:
    if b == "50–69":
        y = S.m_5069.values
        wj = STANDARD[an.JADRO] / STANDARD[an.JADRO].sum()
        pv = [P[P.band == j].set_index("lau").reindex(LAUS) for j in an.JADRO]
        v = sum(wj[j] ** 2 * 1e4 * (t.m / 100) * (1 - t.m / 100) / t.neff for j, t in zip(an.JADRO, pv)).values
    else:
        t = P[P.band == b].set_index("lau").reindex(LAUS)
        y = t.m.values
        v = (1e4 * (t.m / 100) * (1 - t.m / 100) / t.neff).values
    th, se, beta, par, _ = an.fay_herriot(y, v, None, W, X.kraj.values)
    if b == "50–54":
        th0, *_ = an.fay_herriot(y, v * 1e-6, None, W, X.kraj.values)
        assert np.allclose(th0, y, atol=1e-3), "FH při nulovém rozptylu nevrací přímé odhady"
    FH[b] = pd.DataFrame({"prime": y, "se_prime": np.sqrt(v), "fh": th, "se_fh": se}, index=LAUS)
    rows.append({"pásmo": b, "průměr %": y.mean(), "SD mezi okresy (přímé)": y.std(ddof=1),
                 "medián SE přímého odhadu": np.median(np.sqrt(v)), "max SE přímého": np.sqrt(v).max(),
                 "max posun FH p. b.": np.abs(th - y).max(), "λ (prostor)": par["λ"],
                 "σ²_s (prostor)": par["σ_s²"], "σ²_k (kraj)": par["σ_k²"],
                 "podíl rozptylu výběrové chyby": np.mean(v) / y.var(ddof=1)})
FHT = pd.DataFrame(rows).set_index("pásmo")
FHT.to_csv(OUT / "fay_herriot.csv")
FHT.round(3)

,průměr %,SD mezi okresy (přímé),medián SE přímého odhadu,max SE přímého,max posun FH p. b.,λ (prostor),σ²_s (prostor),σ²_k (kraj),podíl rozptylu výběrové chyby
pásmo,,,,,,,,,
45–49,60.947,5.250,0.712,1.297,0.703,0.431,54.951,0.000,0.020
50–54,60.769,4.437,0.782,1.440,0.936,0.592,42.040,0.000,0.034
55–59,59.668,4.831,0.823,1.422,0.727,0.542,49.055,0.158,0.031
60–64,60.698,5.106,0.836,1.413,0.769,0.468,52.498,0.191,0.029
65–69,60.607,5.016,0.789,1.288,0.672,0.426,49.649,0.000,0.027
70–74,55.477,5.585,0.821,1.270,0.631,0.210,49.450,0.000,0.023
75–79,43.942,5.904,0.945,1.495,0.885,0.021,36.408,0.000,0.027
80–84,24.888,5.528,1.048,1.640,0.596,0.103,39.812,0.000,0.037
85+,7.083,2.624,0.633,1.243,0.943,0.196,10.174,0.000,0.068


In [12]:
f69 = FH["50–69"]
fhm = go.Figure()
for i, (col, nazev, sc) in enumerate([("fh", "pokrytí 50–69 let (vyhlazené)", BLUES),
                                      ("se_fh", "směrodatná chyba [p. b.]", BLUES)]):
    fhm.add_trace(go.Choropleth(
        geojson=GEO, locations=LAUS, z=f69[col], colorscale=sc, visible=(i == 0),
        marker_line_color=SURFACE, marker_line_width=0.8,
        customdata=np.c_[X.okres, [cz(v) for v in f69.prime], [cz(v) for v in f69.fh], [cz(v, 2) for v in f69.se_fh]],
        hovertemplate=("<b>%{customdata[0]}</b><br>přímý odhad %{customdata[1]} %<br>vyhlazený "
                       "%{customdata[2]} %<br>SE %{customdata[3]} p. b.<extra></extra>"),
        colorbar=skala(nazev, suffix=" %" if col == "fh" else "")))
fhm.update_layout(**layout("Pokrytí žen 50–69 let: odhad a jeho přesnost",
                           "Fay–Herriot s prostorovým a krajským efektem · přepínač nahoře"))
mapa_geo(fhm)
fhm.update_layout(height=560, margin=dict(l=10, r=10, t=100, b=20),
                  updatemenus=[dict(type="buttons", direction="right", x=1.0, xanchor="right", y=1.12,
                                    bgcolor=SURFACE, bordercolor=AXIS, borderwidth=1,
                                    buttons=[dict(label=l_, method="update", args=[{"visible": [j == i for j in range(2)]}])
                                             for i, l_ in enumerate(["odhad", "přesnost"])])])
fhm.write_html(OUT / "mapa_fay_herriot.html", include_plotlyjs="cdn")
fhm

**Jak to číst.** Výběrová chyba je proti skutečným rozdílům mezi okresy **malá** i v
nejmenších pásmech: model posune odhad nejvýš asi o 1 p. b. a výběrová chyba tvoří jen malou
část rozptylu mezi okresy (sloupec „podíl rozptylu výběrové chyby“). Mapy z revize i z
explorace tedy ukazují skutečné rozdíly, ne šum malých čísel. Prostorová složka je výrazná
u pásem 45–74 (λ ≈ 0,2–0,6), u nejstarších slabší; to potvrzuje shlukování z revize (kap. 5).
Krajský rozptyl (σ²_k) model stáhne k nule u všech pásem kromě 55–64 let, a i tam je proti
prostorovému zanedbatelný: krajský vzorec z revize (kap. 2) zachytí sousedská struktura,
která hranice krajů nekopíruje přesně.

## 7 · Okresy pod a nad očekáváním

Model M3 z revize (socioekonomika, dostupnost, index preventivního chování) předpovídá, jaké
pokrytí by okres „měl mít“. Odchylka od předpovědi ukazuje, kde se děje něco, co model nevidí:
organizace péče, aktivita konkrétních pracovišť a gynekologů, pojišťovny. Pro poctivost bereme
předpověď **bez daného okresu** (leave-one-out) a odchylku měříme studentizovaným reziduem.
Hranice ±1,64 odpovídá 10 % na obou stranách dohromady, při 77 okresech tedy čekáme asi 8 okresů
za hranicí čistě náhodou.

In [13]:
M3 = an.M1 + ["sfca_30", "ma_mamocentrum", "prev_index"]
o3 = sm.OLS(S.m_std.values, sm.add_constant(X[M3].values)).fit()
infl = o3.get_influence()
ODCH = pd.DataFrame({"okres": X.okres, "kraj": X.kraj, "pokrytí %": S.m_std,
                     "předpověď (LOO) %": S.m_std - infl.resid_press,
                     "odchylka p. b.": infl.resid_press, "t": infl.resid_studentized_external}, index=LAUS)
ODCH["skupina"] = np.select([ODCH.t < -1.64, ODCH.t > 1.64], ["pod očekáváním", "nad očekáváním"], "v pásmu")
ODCH.to_csv(OUT / "podvykon.csv")
print(ODCH.skupina.value_counts().to_dict())
ODCH[ODCH.skupina != "v pásmu"].sort_values("t").round(2)

{'v pásmu': 66, 'nad očekáváním': 6, 'pod očekáváním': 5}


,okres,kraj,pokrytí %,předpověď (LOO) %,odchylka p. b.,t,skupina
CZ0802,Frýdek-Místek,Moravskoslezský kraj,47.38,57.50,-10.11,-2.86,pod očekáváním
CZ0522,Jičín,Královéhradecký kraj,50.34,58.73,-8.39,-2.36,pod očekáváním
CZ0711,Jeseník,Olomoucký kraj,38.71,46.49,-7.78,-1.95,pod očekáváním
CZ0801,Bruntál,Moravskoslezský kraj,48.41,55.66,-7.25,-1.89,pod očekáváním
CZ0514,Semily,Liberecký kraj,46.62,52.95,-6.33,-1.78,pod očekáváním
CZ0634,Třebíč,Vysočina,66.72,60.26,6.46,1.75,nad očekáváním
CZ0202,Beroun,Středočeský kraj,57.56,51.23,6.33,1.76,nad očekáváním
CZ0511,Česká Lípa,Liberecký kraj,56.97,50.72,6.25,1.77,nad očekáváním
CZ0532,Pardubice,Pardubický kraj,60.25,53.97,6.28,1.78,nad očekáváním
CZ0712,Olomouc,Olomoucký kraj,64.49,57.80,6.69,1.82,nad očekáváním


In [14]:
om = go.Figure(go.Choropleth(
    geojson=GEO, locations=LAUS, z=ODCH["odchylka p. b."], colorscale=DIV,
    zmid=0, zmin=-10, zmax=10, marker_line_color=SURFACE, marker_line_width=0.8,
    customdata=np.c_[ODCH.okres, [cz(v) for v in ODCH["pokrytí %"]], [cz(v) for v in ODCH["předpověď (LOO) %"]],
                     [cz(v, 2) for v in ODCH.t]],
    hovertemplate=("<b>%{customdata[0]}</b><br>pokrytí %{customdata[1]} %<br>předpověď %{customdata[2]} %"
                   "<br>t = %{customdata[3]}<extra></extra>"),
    colorbar=skala("odchylka od<br>předpovědi", suffix=" p. b.")))
om.add_trace(go.Scattergeo(lon=centra.lon, lat=centra.lat, mode="markers", hoverinfo="skip",
                           marker=dict(size=5, color=INK2, line=dict(color=SURFACE, width=1)), name="pracoviště"))
om.update_layout(**layout("Kde je pokrytí jiné, než by odpovídalo okresu",
                          "standardizované pokrytí 45+ minus předpověď modelu M3 (leave-one-out) · "
                          "modře nad, červeně pod očekáváním"))
mapa_geo(om)
om.update_layout(height=560, showlegend=False, margin=dict(l=10, r=10, t=90, b=20))
om.write_html(OUT / "mapa_podvykon.html", include_plotlyjs="cdn")
om

**Jak to číst.** Za hranicí je 11 okresů, o něco víc než ~8, které by dala náhoda, takže
jednotlivé okresy neber jako jistý nález. Užitečné je, **kde** leží: pod očekáváním vycházejí Jeseník a
Bruntál (kde i po opravě kap. 1 jezdí polovina žen jinam), Frýdek-Místek, Jičín a Semily; nad
očekáváním hlavně okresy střední Moravy a Vysočiny (Kroměříž, Olomouc, Třebíč). To jsou kandidáti
na **kvalitativní srovnání**: co dělají pracoviště a gynekologové v Kroměříži jinak než ve
Frýdku-Místku? Takové srovnání žádná okresní data nenahradí.

## 8 · Shrnutí

In [15]:
r_doj = stats.pearsonr(X.dojizdka_kor, S.m_std)
r_c30 = KAP[(KAP["míra"] == "pracoviště v dosahu") & (KAP["dosah km"] == 30) & (KAP["cíl"] == "standardizované 45+")].iloc[0]
r_v30 = KAP[(KAP["míra"] == "provedená vyšetření v dosahu") & (KAP["dosah km"] == 30) & (KAP["cíl"] == "standardizované 45+")].iloc[0]
g_m = GYN[(GYN["nabídka"] == POPIS["gyn_ord_10k"][0]) & (GYN["s čím"] == "pokrytí 45+ (std.)")].iloc[0]
SHRNUTI = pd.DataFrame([
    ("Dojížďka po opravě vykazování řetězců",
     f"7 okresů z 100 % na 5–51 %; vztah s pokrytím drží, r = {cz(r_doj[0], 2)} (p = {cz(r_doj[1], 3)})",
     "dojížďka souvisí s nižším pokrytím"),
    ("Kapacita pracovišť",
     f"pracoviště v dosahu 30 km r = {cz(r_c30.r, 2)}; provedená vyšetření v dosahu r = {cz(r_v30.r, 2)} (kruhové)",
     "počet pracovišť nerozhoduje; kapacitu od zájmu data neoddělí"),
    ("Hustota gynekologů",
     f"ordinace na 10 000 žen × pokrytí r = {cz(g_m.r, 2)} (p = {cz(g_m.p, 2)}); mediace nulová",
     "rozhoduje, co gynekologové dělají, ne kolik jich je"),
    ("Otevření pracoviště (Teplice 2016)",
     f"rozdíl proti kraji se zavřel o {cz(efekt_tep)} p. b.; placebo p = {cz(p_placebo, 2)}",
     "slabý doklad, jediný případ"),
    ("Pojišťovny", "okresní ani krajská data o pojištěncích podle pojišťovny nejsou otevřená", "datová mezera"),
    ("Stadium při diagnóze",
     f"celostátně III–IV {cz(narod.pozde_pct.loc[1995])} % (1995) → {cz(narod.pozde_pct.loc[2024])} % (2024); "
     f"mezi kraji p = {cz(fe.pvalues['m2_pred'], 2)}",
     "celostátní trend ano, krajský vztah nelze prokázat"),
    ("Přesnost okresních odhadů",
     f"max posun Fay–Herriot {cz(FHT['max posun FH p. b.'].max())} p. b.",
     "rozdíly mezi okresy jsou skutečné, ne šum"),
    ("Okresy mimo očekávání",
     f"pod: {', '.join(ODCH[ODCH.skupina == 'pod očekáváním'].okres)}; "
     f"nad: {', '.join(ODCH[ODCH.skupina == 'nad očekáváním'].okres)}",
     "kandidáti na kvalitativní srovnání"),
], columns=["otázka", "výsledek", "závěr"])
SHRNUTI.to_csv(OUT / "shrnuti.csv", index=False)
with pd.option_context("display.max_colwidth", 200):
    display(SHRNUTI.style.hide(axis="index").set_properties(**{"text-align": "left"}))

otázka,výsledek,závěr
Dojížďka po opravě vykazování řetězců,"7 okresů z 100 % na 5–51 %; vztah s pokrytím drží, r = -0,30 (p = 0,008)",dojížďka souvisí s nižším pokrytím
Kapacita pracovišť,"pracoviště v dosahu 30 km r = 0,02; provedená vyšetření v dosahu r = 0,32 (kruhové)",počet pracovišť nerozhoduje; kapacitu od zájmu data neoddělí
Hustota gynekologů,"ordinace na 10 000 žen × pokrytí r = 0,05 (p = 0,64); mediace nulová","rozhoduje, co gynekologové dělají, ne kolik jich je"
Otevření pracoviště (Teplice 2016),"rozdíl proti kraji se zavřel o 2,2 p. b.; placebo p = 0,09","slabý doklad, jediný případ"
Pojišťovny,okresní ani krajská data o pojištěncích podle pojišťovny nejsou otevřená,datová mezera
Stadium při diagnóze,"celostátně III–IV 28,8 % (1995) → 13,3 % (2024); mezi kraji p = 0,45","celostátní trend ano, krajský vztah nelze prokázat"
Přesnost okresních odhadů,"max posun Fay–Herriot 0,9 p. b.","rozdíly mezi okresy jsou skutečné, ne šum"
Okresy mimo očekávání,"pod: Semily, Jičín, Jeseník, Bruntál, Frýdek-Místek; nad: Beroun, Česká Lípa, Pardubice, Třebíč, Olomouc, Kroměříž",kandidáti na kvalitativní srovnání


## Metodické poznámky

1. **Rozdělení IČZ** (kap. 1) je heuristika: v 7 okresech s pracovištěm řetězce, které nevykazuje
   samo, bereme IČZ s nejvíce ženami z okresu jako místní pracoviště. Přesné řešení by potřebovalo
   převodník IČZ → adresa pracoviště, který otevřená data nemají.
2. **NRPZS** obsahuje jen dnes fungující místa péče, takže hustota gynekologů je průřez 10/2026,
   porovnávaný s pokrytím 2020–2024.
3. **Historie pracovišť** je ze snímků web.archive.org (2008–2011, 2013, 2015–2019) a dnešního
   seznamu. Roky 2012, 2014 a 2020–2025 chybí. Seznam uvádí akreditaci, ne provoz.
4. **NOR** v otevřených datech nemá okresy, jen kraje. Podíl neznámého stadia je 2–8 % podle kraje.
5. **Fay–Herriot** bere výběrový rozptyl jako binomický z průměrného ročního počtu žen v pásmu a
   směrodatná chyba nezahrnuje nejistotu odhadu parametrů rozptylu.
6. Všechny vztahy jsou **ekologické**.